
# 02_03 Feature space — clean (timing & `time_df`)

Self-contained notebook for **AST index time** and **diagnostic analyses**.

## Cohort rules (this notebook)

1. **Respiratory cohort** — same `hadm_id` filter as upstream (`resp_ast_df`).
2. **Exclude the entire hospital admission** if **any** AST row in `resp_ast_df` has `charttime` **strictly before** `first_icu_admit_time` (earliest ICU `intime` for that `hadm_id`). No pre‑ICU cultures on the index admission.
3. **Index culture** — among admissions that pass (2), take the **earliest** `charttime` with `charttime >= first_icu_admit_time` (**first AST on/after first ICU admit**).
4. **`stay_id` / ICU window** — the ICU stay whose `[intime, outtime]` contains that sample when possible; if the sample falls **between** ICU stays (ward gap), `stay_id` is left missing and flagged (`ast_inside_icu_stay=False`).

Outputs: `time_df` saved via `hh.parq(time_df, "time_df_after_first_icu")` plus printed summaries.

**Prior admissions (features, not index):** In `02_4_Feature_space_creation_clean.ipynb`, `get_prior_events` attaches **all** events for the same `subject_id` with event time **strictly before** the index admission’s `hospital_admit_time` (prior `hadm_id`s only; no same‑admission leakage).

**Related:** Full ETL lives in `02_03_Feature_space.ipynb`; this file is for reproducible timing logic and QC plots/tables.


## Setup


In [1]:

import pandas as pd
import numpy as np
from pathlib import Path

import tools.helpers as hh

DATA_ROOT = Path("/Users/gnaanikko.pa/Documents/Academic /MIMIC/model_building")
PARQ_DIR = DATA_ROOT / "parq"
OUTPUT_DIR = DATA_ROOT / "outputs"

# Cohort + AST extract (match paths in 02_03)
RESP_COHORT_PATH = PARQ_DIR / "resp_inf_cohort_23Jan26_1953.parquet"
ALL_AST_PATH = PARQ_DIR / "all_ast_df27Feb26_1751.parquet"
# Optional: legacy time_df for before/after row counts
LEGACY_TIME_DF_GLOB = "time_df*.parquet"


## Load data


In [2]:

resp_inf_cohort = pd.read_parquet(RESP_COHORT_PATH)
all_ast_df = pd.read_parquet(ALL_AST_PATH)

resp_ast_df = hh.df_subset(all_ast_df, resp_inf_cohort, by_col="hadm_id")
resp_ast_df = resp_ast_df.copy()
resp_ast_df["charttime"] = pd.to_datetime(resp_ast_df["charttime"])

icu_stays_df = resp_inf_cohort[
    ["hadm_id", "stay_id", "icu_admit_time", "icu_discharge_time"]
].drop_duplicates()

print("resp_ast_df:", resp_ast_df.shape)
print("icu stay rows:", len(icu_stays_df), "| admissions:", icu_stays_df["hadm_id"].nunique())


resp_ast_df: (127899, 20)
icu stay rows: 16438 | admissions: 13611



## Analysis A — Legacy `time_df` (optional)

If a parquet from the **old** merge-on-`stay_id` pipeline exists, load it and summarise `hours_icu_to_ast` (negative ⇒ sample before that row’s `icu_admit_time`).


In [3]:

legacy_paths = sorted(PARQ_DIR.glob(LEGACY_TIME_DF_GLOB), key=lambda p: p.stat().st_mtime, reverse=True)
legacy_time_df = None
for p in legacy_paths:
    if "corrected" in p.name.lower() or "after_first" in p.name.lower():
        continue
    try:
        legacy_time_df = pd.read_parquet(p)
        if "hours_icu_to_ast" in legacy_time_df.columns and "first_ast_time" in legacy_time_df.columns:
            print("Loaded legacy:", p.name)
            break
    except Exception:
        continue

if legacy_time_df is not None:
    n = len(legacy_time_df)
    print(f"Legacy rows: {n}")
    print("AST before ICU admit (hours_icu_to_ast < 0):", int((legacy_time_df["hours_icu_to_ast"] < 0).sum()))
    print("AST after ICU admit:", int((legacy_time_df["hours_icu_to_ast"] >= 0).sum()))
    if "icu_discharge_time" in legacy_time_df.columns:
        inside = (
            (legacy_time_df["first_ast_time"] >= legacy_time_df["icu_admit_time"])
            & (legacy_time_df["first_ast_time"] <= legacy_time_df["icu_discharge_time"])
        )
        print("AST inside [icu_admit, icu_discharge] for listed stay:", int(inside.sum()))
else:
    print("No legacy time_df parquet matched; skip comparison.")


Loaded legacy: time_df05Mar26_2130.parquet
Legacy rows: 4426
AST before ICU admit (hours_icu_to_ast < 0): 865
AST after ICU admit: 3561
AST inside [icu_admit, icu_discharge] for listed stay: 2935



## Analysis B — First AST per admission vs “inside any ICU stay”

- **First AST (naive):** `groupby(hadm_id) charttime` minimum on `resp_ast_df` (no pre-ICU filter).
- **Inside ICU:** that timestamp falls in `[icu_admit_time, icu_discharge_time]` for **some** ICU row of the same `hadm_id`.


In [4]:

first_ast_naive = resp_ast_df.loc[resp_ast_df.groupby("hadm_id")["charttime"].idxmin()][
    ["hadm_id", "charttime"]
].rename(columns={"charttime": "first_ast_time"})

ast_x_stays = first_ast_naive.merge(icu_stays_df, on="hadm_id", how="inner")
ast_x_stays["inside_this_stay"] = (
    (ast_x_stays["first_ast_time"] >= ast_x_stays["icu_admit_time"])
    & (ast_x_stays["first_ast_time"] <= ast_x_stays["icu_discharge_time"])
)
inside_any = ast_x_stays.groupby("hadm_id")["inside_this_stay"].any()

n_naive = first_ast_naive["hadm_id"].nunique()
n_inside = int(inside_any.sum())
print("First AST per hadm (naive, no pre-ICU filter):", n_naive)
print("… of which fall inside SOME ICU interval:", n_inside)
print("… excluded (not inside any ICU):", n_naive - n_inside)


First AST per hadm (naive, no pre-ICU filter): 4426
… of which fall inside SOME ICU interval: 3311
… excluded (not inside any ICU): 1115



## Analysis C — Multiple ICU stays: samples **between** stays

For admissions with ≥2 ICU stays, flag sample times strictly **after** `icu_discharge` of stay *k* and **before** `icu_admit` of stay *k+1*.


In [5]:

def ast_between_icu_stays(hadm_id: int, ast_time: pd.Timestamp, icu_stays: pd.DataFrame) -> bool:
    stays = icu_stays[icu_stays["hadm_id"] == hadm_id].sort_values("icu_admit_time")
    if len(stays) < 2:
        return False
    if ((ast_time >= stays["icu_admit_time"]) & (ast_time <= stays["icu_discharge_time"])).any():
        return False
    ends = stays["icu_discharge_time"].tolist()
    starts = stays["icu_admit_time"].tolist()
    for i in range(len(starts) - 1):
        if ast_time > ends[i] and ast_time < starts[i + 1]:
            return True
    return False

multi_hadm = icu_stays_df.groupby("hadm_id").size()
multi_hadm = multi_hadm[multi_hadm >= 2].index

sample_times = (
    resp_ast_df[resp_ast_df["hadm_id"].isin(multi_hadm)]
    .groupby(["hadm_id", "charttime"])
    .size()
    .reset_index()[["hadm_id", "charttime"]]
)
between_mask = sample_times.apply(
    lambda r: ast_between_icu_stays(r["hadm_id"], r["charttime"], icu_stays_df),
    axis=1,
)
between_samples = sample_times[between_mask]
print("Admissions with 2+ ICU stays:", len(multi_hadm))
print("Distinct (hadm_id, charttime) in gap between ICU stays:", len(between_samples))
print("Distinct hadm_id with ≥1 such sample:", between_samples["hadm_id"].nunique())

first_ast_multi = first_ast_naive[first_ast_naive["hadm_id"].isin(multi_hadm)]
first_in_gap = first_ast_multi.apply(
    lambda r: ast_between_icu_stays(r["hadm_id"], r["first_ast_time"], icu_stays_df),
    axis=1,
)
print(
    "Among multi-ICU admissions: first AST (naive) falls in gap:",
    int(first_in_gap.sum()),
    "/",
    len(first_ast_multi),
)


Admissions with 2+ ICU stays: 2227
Distinct (hadm_id, charttime) in gap between ICU stays: 489
Distinct hadm_id with ≥1 such sample: 304
Among multi-ICU admissions: first AST (naive) falls in gap: 184 / 1077



## Analysis D — “Strict inside ICU” vs “after first ICU admit”

- **Strict:** first AST that lies inside **some** ICU window (earliest qualifying `charttime`).
- **This notebook’s policy:** first AST with `charttime >= first_icu_admit_time` (may still sit in a ward gap).


In [6]:

first_icu_admit = icu_stays_df.groupby("hadm_id", as_index=False)["icu_admit_time"].min()
first_icu_admit = first_icu_admit.rename(columns={"icu_admit_time": "first_icu_admit_time"})

ra = resp_ast_df.merge(first_icu_admit, on="hadm_id")
after_icu = ra[ra["charttime"] >= ra["first_icu_admit_time"]]
n_after_policy = after_icu.groupby("hadm_id").ngroups
print("Admissions with ≥1 AST on/after first ICU admit:", n_after_policy)

# Strict: expand all AST x stays, keep inside, earliest per hadm
expanded = resp_ast_df.merge(icu_stays_df, on="hadm_id")
expanded["inside"] = (expanded["charttime"] >= expanded["icu_admit_time"]) & (
    expanded["charttime"] <= expanded["icu_discharge_time"]
)
inside_ast = expanded[expanded["inside"]].sort_values(["hadm_id", "charttime"])
strict_first = inside_ast.groupby("hadm_id", as_index=False).first()
print("Admissions with strict first-AST-inside-ICU:", len(strict_first))

naive_ids = set(first_ast_naive["hadm_id"])
strict_ids = set(strict_first["hadm_id"])
lost_from_naive_inside = naive_ids - strict_ids
print("Naive-first hadms with NO qualifying inside-ICU AST:", len(lost_from_naive_inside))


Admissions with ≥1 AST on/after first ICU admit: 4062
Admissions with strict first-AST-inside-ICU: 3501
Naive-first hadms with NO qualifying inside-ICU AST: 925



## Build `time_df` — first AST on/after first ICU admission

Steps:

1. `first_icu_admit_time` = minimum `icu_admit_time` per `hadm_id`.
2. **Drop entire `hadm_id`** if any AST has `charttime < first_icu_admit_time`.
3. Among remaining admissions, keep AST rows with `charttime >= first_icu_admit_time` and take **earliest** `charttime` per `hadm_id`.
4. If the sample lies inside an ICU interval, set `stay_id` / `icu_admit_time` / `icu_discharge_time` from that stay. If not (ward gap), **do not** copy a misleading stay from the raw AST row: leave `stay_id` as NaN and keep `first_icu_admit_time` for timing; `hours_icu_to_ast` uses matched `icu_admit_time` when present, else `first_icu_admit_time`.


In [10]:

first_icu_admit = icu_stays_df.groupby("hadm_id", as_index=False)["icu_admit_time"].min()
first_icu_admit = first_icu_admit.rename(columns={"icu_admit_time": "first_icu_admit_time"})

ra_all = resp_ast_df.merge(first_icu_admit, on="hadm_id")
# Exclude entire admission if ANY culture is drawn before first ICU intime
hadm_any_pre_icu_ast = (
    ra_all.loc[ra_all["charttime"] < ra_all["first_icu_admit_time"], "hadm_id"]
    .drop_duplicates()
)
print("hadm_id excluded (any AST before first ICU):", len(hadm_any_pre_icu_ast))

eligible = ra_all[
    (~ra_all["hadm_id"].isin(hadm_any_pre_icu_ast))
    & (ra_all["charttime"] >= ra_all["first_icu_admit_time"])
].copy()

idx = eligible.groupby("hadm_id")["charttime"].idxmin()
first_rows = eligible.loc[idx].reset_index(drop=True)
first_rows = first_rows.rename(columns={"charttime": "first_ast_time"})

ast_expanded = first_rows[["hadm_id", "first_ast_time"]].merge(icu_stays_df, on="hadm_id")
ast_expanded["inside"] = (ast_expanded["first_ast_time"] >= ast_expanded["icu_admit_time"]) & (
    ast_expanded["first_ast_time"] <= ast_expanded["icu_discharge_time"]
)
containing = (
    ast_expanded[ast_expanded["inside"]]
    .sort_values(["hadm_id", "icu_admit_time"])
    .groupby("hadm_id", as_index=False)
    .first()
)
containing = containing[
    ["hadm_id", "stay_id", "icu_admit_time", "icu_discharge_time"]
].rename(
    columns={
        "stay_id": "matched_stay_id",
        "icu_admit_time": "icu_admit_time_matched",
        "icu_discharge_time": "icu_discharge_time_matched",
    }
)

time_df = first_rows.merge(containing, on="hadm_id", how="left")
time_df["ast_inside_icu_stay"] = time_df["matched_stay_id"].notna()
time_df["stay_id"] = time_df["matched_stay_id"]
time_df["icu_admit_time"] = time_df["icu_admit_time_matched"]
time_df["icu_discharge_time"] = time_df["icu_discharge_time_matched"]

hadm_demo = (
    resp_inf_cohort.sort_values(["hadm_id", "icu_admit_time"])
    .groupby("hadm_id", as_index=False)
    .first()[
        [
            "subject_id",
            "hadm_id",
            "hospital_admit_time",
            "hospital_discharge_time",
            "hospital_death_time",
        ]
    ]
)

time_df = time_df.drop(
    columns=[
        "matched_stay_id",
        "icu_admit_time_matched",
        "icu_discharge_time_matched",
    ],
    errors="ignore",
)
time_df = time_df.merge(hadm_demo, on=["hadm_id", "subject_id"], how="left")

# Hours relative to hospital admit; ICU offset uses matched ICU admit when inside, else first ICU admit
icu_ref_admit = time_df["icu_admit_time"].fillna(time_df["first_icu_admit_time"])
time_df["hours_adm_to_ast"] = (
    time_df["first_ast_time"] - time_df["hospital_admit_time"]
).dt.total_seconds() / 3600
time_df["hours_icu_to_ast"] = (
    time_df["first_ast_time"] - icu_ref_admit
).dt.total_seconds() / 3600
time_df["hours_first_icu_admit_to_ast"] = (
    time_df["first_ast_time"] - time_df["first_icu_admit_time"]
).dt.total_seconds() / 3600

out_cols = [
    "subject_id",
    "hadm_id",
    "stay_id",
    "hospital_admit_time",
    "hospital_discharge_time",
    "hospital_death_time",
    "icu_admit_time",
    "icu_discharge_time",
    "first_icu_admit_time",
    "first_ast_time",
    "ast_inside_icu_stay",
    "hours_adm_to_ast",
    "hours_icu_to_ast",
    "hours_first_icu_admit_to_ast",
]
time_df = time_df[[c for c in out_cols if c in time_df.columns]]

print("time_df rows:", len(time_df))
print("ast_inside_icu_stay True:", int(time_df["ast_inside_icu_stay"].sum()))
print("ast_inside_icu_stay False:", int((~time_df["ast_inside_icu_stay"]).sum()))
print("hours_first_icu_admit_to_ast >= 0:", int((time_df["hours_first_icu_admit_to_ast"] >= 0).sum()), "/", len(time_df))


hadm_id excluded (any AST before first ICU): 544
time_df rows: 3882
ast_inside_icu_stay True: 3311
ast_inside_icu_stay False: 571
hours_first_icu_admit_to_ast >= 0: 3882 / 3882


## Save


In [11]:

hh.dxx(time_df)
hh.parq(time_df, "time_df_after_first_icu")


3.6k Unique Patient IDs (3588)
3.9k Unique Admission IDs (3882)
3.3k Unique ICU Stay IDs (3311)
3.9k Rows, shape: (3882, 14)



,subject_id,hadm_id,stay_id,hospital_admit_time,hospital_discharge_time,hospital_death_time,icu_admit_time,icu_discharge_time,first_icu_admit_time,first_ast_time,ast_inside_icu_stay,hours_adm_to_ast,hours_icu_to_ast,hours_first_icu_admit_to_ast
dtype,int64,int64,float64,datetime64[ns],datetime64[ns],datetime64[ns],datetime64[ns],datetime64[ns],datetime64[ns],datetime64[ns],bool,float64,float64,float64
NotNA | NA,3882 | 0,3882 | 0,3311 | 571,3882 | 0,3882 | 0,915 | 2967,3311 | 571,3311 | 571,3882 | 0,3882 | 0,3882 | 0,3882 | 0,3882 | 0,3882 | 0
nunique,3588,3882,3312,3880,3880,916,3312,3312,3882,3882,2,3264,3655,3657
0,18172155,20009330,36841282.000000,2144-01-01 00:33:00,2144-01-09 21:07:00,NaT,2144-01-01 04:26:36,2144-01-06 22:46:18,2144-01-01 04:26:36,2144-01-01 11:53:00,True,11.333333,7.440000,7.440000
1,19318312,20013496,32840324.000000,2161-11-04 04:33:00,2161-12-01 17:15:00,NaT,2161-11-04 06:58:00,2161-11-22 18:27:35,2161-11-04 06:58:00,2161-11-04 22:30:00,True,17.950000,15.533333,15.533333
2,17333498,20015712,nan,2162-05-14 03:54:00,2162-06-11 13:55:00,NaT,NaT,NaT,2162-05-14 04:28:00,2162-05-22 16:00:00,False,204.100000,203.533333,203.533333


File saved at: time_df_after_first_icu12Apr26_1615.parquet


### Cohort-aligned AST — before / after `time_df`

`resp_ast_cohort` and `all_ast_cohort` restrict micro rows to **`hadm_id`** in **`time_df`**. **`index_ast_rows`** keeps only lines where **`charttime`** equals **`first_ast_time`** (index culture; **`stay_id`** not required).


In [ ]:
print("=== Before restricting to time_df cohort ===")
print(
    "resp_ast_df  rows:", len(resp_ast_df),
    "| hadm_id nunique:", resp_ast_df["hadm_id"].nunique(),
)
print(
    "all_ast_df   rows:", len(all_ast_df),
    "| hadm_id nunique:", all_ast_df["hadm_id"].nunique(),
)
print("time_df rows:", len(time_df), "| hadm_id nunique:", time_df["hadm_id"].nunique())

resp_ast_cohort = hh.df_subset(resp_ast_df, isin_df=time_df, by_col="hadm_id")
all_ast_cohort = hh.df_subset(all_ast_df, isin_df=time_df, by_col="hadm_id")

print("=== After: AST ∩ time_df (hadm_id) ===")
print(
    "resp_ast_cohort rows:", len(resp_ast_cohort),
    "| hadm_id nunique:", resp_ast_cohort["hadm_id"].nunique(),
)
print(
    "all_ast_cohort  rows:", len(all_ast_cohort),
    "| hadm_id nunique:", all_ast_cohort["hadm_id"].nunique(),
)

_tm = time_df[["hadm_id", "first_ast_time"]].drop_duplicates("hadm_id")
_idx = resp_ast_cohort.merge(_tm, on="hadm_id", how="inner")
_idx["charttime"] = pd.to_datetime(_idx["charttime"])
_idx["first_ast_time"] = pd.to_datetime(_idx["first_ast_time"])
index_ast_rows = _idx[_idx["charttime"] == _idx["first_ast_time"]].copy()

print("=== Index culture (charttime == first_ast_time) ===")
print(
    "index_ast_rows rows:", len(index_ast_rows),
    "| hadm_id nunique:", index_ast_rows["hadm_id"].nunique(),
)
_miss = set(time_df["hadm_id"]) - set(index_ast_rows["hadm_id"])
print("time_df hadm_id with no matching index row:", len(_miss))
if len(_miss) and len(_miss) <= 15:
    print("  hadm_id:", sorted(_miss))
